## Problem Statement

### Context

AllLife Bank is a US bank that has a growing customer base. The majority of these customers are liability customers (depositors) with varying sizes of deposits. The number of customers who are also borrowers (asset customers) is quite small, and the bank is interested in expanding this base rapidly to bring in more loan business and in the process, earn more through the interest on loans. In particular, the management wants to explore ways of converting its liability customers to personal loan customers (while retaining them as depositors).

A campaign that the bank ran last year for liability customers showed a healthy conversion rate of over 9% success. This has encouraged the retail marketing department to devise campaigns with better target marketing to increase the success ratio.

You as a Data scientist at AllLife bank have to build a model that will help the marketing department to identify the potential customers who have a higher probability of purchasing the loan.

### Objective

To predict whether a liability customer will buy personal loans, to understand which customer attributes are most significant in driving purchases, and identify which segment of customers to target more.

### Data Dictionary
* `ID`: Customer ID
* `Age`: Customer’s age in completed years
* `Experience`: #years of professional experience
* `Income`: Annual income of the customer (in thousand dollars)
* `ZIP Code`: Home Address ZIP code.
* `Family`: the Family size of the customer
* `CCAvg`: Average spending on credit cards per month (in thousand dollars)
* `Education`: Education Level. 1: Undergrad; 2: Graduate;3: Advanced/Professional
* `Mortgage`: Value of house mortgage if any. (in thousand dollars)
* `Personal_Loan`: Did this customer accept the personal loan offered in the last campaign? (0: No, 1: Yes)
* `Securities_Account`: Does the customer have securities account with the bank? (0: No, 1: Yes)
* `CD_Account`: Does the customer have a certificate of deposit (CD) account with the bank? (0: No, 1: Yes)
* `Online`: Do customers use internet banking facilities? (0: No, 1: Yes)
* `CreditCard`: Does the customer use a credit card issued by any other Bank (excluding All life Bank)? (0: No, 1: Yes)

## Importing necessary libraries

In [ ]:
# Installing the libraries with the specified version.
!pip install numpy==2.0.2 pandas==2.2.2 matplotlib==3.10.0 seaborn==0.13.2 scikit-learn==1.6.1 -q sklearn-pandas==2.2.0 -q --user

In [ ]:
# Libraries to help with reading and manipulating data
import pandas as pd
import numpy as np

# libaries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Library to split data
from sklearn.model_selection import train_test_split

# To build model for prediction
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree

# To get diferent metric scores
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    recall_score,
    precision_score,
    confusion_matrix,
)

# to suppress unnecessary warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Loading the dataset

In [ ]:
loan = pd.read_csv("/content/drive/MyDrive/Loan_Modelling.csv")

## Data Overview

* Observations
* Sanity checks

#Observations


In [ ]:
loan.info()

In [ ]:
loan.head()

In [ ]:
loan.tail()

In [ ]:
loan.shape

In [ ]:
loan.describe().T

In [ ]:
#The code to drop a Product ID column from the dataframe
loan = loan.drop(["ID"], axis=1)

The ID column does not add much value for data analysis or affects the target variable.

In [ ]:
loan["Age"].unique()

In [ ]:
#Experience column has negative values and hence we treat them and convert them to positive values.
loan["Experience"].unique()

In [ ]:
loan[loan["Experience"] < 0]["Experience"].unique()
# Removing the negative values for experience
loan["Experience"].replace(-1, 1, inplace=True)
loan["Experience"].replace(-2, 2, inplace=True)
loan["Experience"].replace(-3, 3, inplace=True)

Removing negative values and replacing them with positive numbers will help in our EDA.

In [ ]:
loan["Education"].unique()

## Data Preprocessing

* Missing value treatment
* Feature engineering (if needed)
* Outlier detection and treatment (if needed)
* Preparing data for modeling
* Any other preprocessing steps (if needed)

In [ ]:
#Checking for missing values and treatment, if any.
loan.isna().sum()

There are no missing values in the above data.

##Feature Engineering

In [ ]:
# checking the number of uniques in the zip code
loan["ZIPCode"].nunique()

In [ ]:
loan["ZIPCode"] = loan["ZIPCode"].astype(str)
print("Number of unique values if we take first two digits of ZIPCode: ",
    loan["ZIPCode"].str[0:2].nunique(),
)
loan["ZIPCode"] = loan["ZIPCode"].str[0:2]

loan["ZIPCode"] = loan["ZIPCode"].astype("category")

In [ ]:
loan["ZIPCode"].unique()

Typically, Zipcode is assigned to a region and in the given data , it is number. We extract first 2 digits to understand the region part of it and hen make it category to be more relevant or useful for our analysis. Two digit region code will not be as useful as category for our further analysis.

## Exploratory Data Analysis.

- EDA is an important part of any project involving data.
- It is important to investigate and understand the data better before building a model with it.
- A few questions have been mentioned below which will help you approach the analysis in the right manner and generate insights from the data.
- A thorough analysis of the data, in addition to the questions mentioned below, should be done.

**Questions**:

1. What is the distribution of mortgage attribute? Are there any noticeable patterns or outliers in the distribution?
2. How many customers have credit cards?
3. What are the attributes that have a strong correlation with the target attribute (personal loan)?
4. How does a customer's interest in purchasing a loan vary with their age?
5. How does a customer's interest in purchasing a loan vary with their education?

##Univariate Analysis

In [ ]:
def histogram_boxplot(data, feature, figsize=(12, 7), kde=False, bins=None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (12,7))
    kde: whether to show the density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2,  # Number of rows of the subplot grid= 2
        sharex=True,  # x-axis will be shared among all subplots
        gridspec_kw={"height_ratios": (0.25, 0.75)},
        figsize=figsize,
    )  # creating the 2 subplots
    sns.boxplot(
        data=data, x=feature, ax=ax_box2, showmeans=True, color="violet"
    )  # boxplot will be created and a star will indicate the mean value of the column
    sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2, bins=bins, palette="winter"
    ) if bins else sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2
    )  # For histogram
    ax_hist2.axvline(
        data[feature].mean(), color="green", linestyle="--"
    )  # Add mean to the histogram
    ax_hist2.axvline(
        data[feature].median(), color="black", linestyle="-"
    )  # Add median to the histogram

In [ ]:
# function to create labeled barplots
def labeled_barplot(data, feature, perc=False, n=None):
    """
    Barplot with percentage at the top

    data: dataframe
    feature: dataframe column
    perc: whether to display percentages instead of count (default is False)
    n: displays the top n category levels (default is None, i.e., display all levels)
    """

    total = len(data[feature])  # length of the column
    count = data[feature].nunique()
    if n is None:
        plt.figure(figsize=(count + 1, 5))
    else:
        plt.figure(figsize=(n + 1, 5))

    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=data,
        x=feature,
        palette="Paired",
        order=data[feature].value_counts().index[:n].sort_values(),
    )

    for p in ax.patches:
        if perc == True:
            label = "{:.1f}%".format(
                100 * p.get_height() / total
            )  # percentage of each class of the category
        else:
            label = p.get_height()  # count of each level of the category

        x = p.get_x() + p.get_width() / 2  # width of the plot
        y = p.get_height()  # height of the plot

        ax.annotate(
            label,
            (x, y),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",
        )  # annotate the percentage

    plt.show()  # show the plot

In [ ]:
#Observations on Age
histogram_boxplot(loan,"Age")

Age for customers ranges mainly from 30-65 years , with average around 45-46 years of age.
There are no outliers for age.

In [ ]:
#Observations on Experience
histogram_boxplot(loan,"Experience")

*  There are higher number of people between 7-8 years 15 years , 25 years and 35 years of work experience. There are no outliers in this data.

In [ ]:
histogram_boxplot(loan,"Income")

*  Range of income is more between 45K to 55K and between 60K to 75K .
*  Data is right skewed.
*  There are also some number of outliers.






In [ ]:
histogram_boxplot(loan,"CCAvg")

- CCAvg seem to have right skewed data and a number of outliers .

In [ ]:
histogram_boxplot(loan,"Mortgage")

*  Large number of people who have not taken mortgage yet and hence have 0 values.
*  There are also large number of continuous outliers seen in Mortgage data, where customers have taken mortgage around 100K and above.



In [ ]:
labeled_barplot(loan, "Family", perc=True)

*  We observe that 29.4% of customers are single, followed by 25.9% with two and 24.4% with 4 family members.




In [ ]:
labeled_barplot(loan,"Education",perc=True)

* 41.9% of customers are UnderGrad, 28.1% are graduates and 30.0% are Advanced level graduates.



In [ ]:
labeled_barplot(loan,"Securities_Account",perc=True)

*  89.6% Customers do not have Securities account whereas 10.4% do have the account.



In [ ]:
labeled_barplot(loan,"CD_Account",perc=True)

*  About 94% customers do not have Certificates of Deposit(CD) whereas 6% have CD account  with the bank .



In [ ]:
labeled_barplot(loan,"Online",perc=True)   ## Complete the code to create labeled_barplot for Online

*  We see that 59.7% f customer use online banking services whereas 40.3% do not use these services.



In [ ]:
labeled_barplot(loan,"CreditCard",perc=True)   ## Complete the code to create labeled_barplot for Credit Cards

*  70.6% Customers do not have credit cards and only 29.4% of customers do use credit card issued by the bank.



Questions(with Answers):

1. What is the distribution of mortgage attribute? Are there any noticeable patterns or outliers in the distribution?
-  Mortgage value avergae value is 56000 USD. Maximum mortgage taken is 635000 USD. And there are 5000 people who have taken mortgage.
- The lower quartile in boxplot has lower quartile close to 0 value which means data has large values of customers who have not taken mortgage. And also there are large number of continuous outliers.

2. How many customers have credit cards?
- From the data , we see 70.6% Customers do not have credit cards and only 29.4% of customers do use credit card issued by the bank.

3. What are the attributes that have a strong correlation with the target attribute (personal loan)?
- From the analysis below, we see the following points.
 - We see correlation between Personal loan and income of the individual.
 - We see correlation between CCAvg and Personal loan. Customer's ability to take personal loan is also dependent on average spending on credit cards per month.
 - We also see a correlation between Family members and Personal loan taken by the customers. Personal loan taken with more family members like 3 or 4 is higher.
 - There is a some correlation between customers who have Certificates of deposit account with the bank and taking a personal loan. This might be because customers have some safety aspect to repay personal loan amount.

4. How does a customer's interest in purchasing a loan vary with their age?
- We see based on the below visualization , we see the highest number of customers who take personal loan are between the ages 30-39 followed by 50-59 and 40-49 age groups. This shows that customer's interest in purchasing loan is higher in a certain age bracket and most likely due to the source of income.

5. How does a customer's interest in purchasing a loan vary with their education?
- Customers who have advanced or professional degree(Education) are more likely to take a personal loan.
Customers with graduate degree seem to be the next who take personal load and the ones with Undergrad education seem to be the last in numbers to take personal loan.

##Bivariate Analysis

*  Common methods to be used to avoid redundant code for each variable.



In [ ]:
def stacked_barplot(data, predictor, target):
    """
    Print the category counts and plot a stacked bar chart

    data: dataframe
    predictor: independent variable
    target: target variable
    """
    count = data[predictor].nunique()
    sorter = data[target].value_counts().index[-1]
    tab1 = pd.crosstab(data[predictor], data[target], margins=True).sort_values(
        by=sorter, ascending=False
    )
    print(tab1)
    print("-" * 120)
    tab = pd.crosstab(data[predictor], data[target], normalize="index").sort_values(
        by=sorter, ascending=False
    )
    tab.plot(kind="bar", stacked=True, figsize=(count + 5, 5))
    plt.legend(
        loc="lower left", frameon=False,
    )
    plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
    plt.show()

In [ ]:
# function to plot distributions wrt target
def distribution_plot_wrt_target(data, predictor, target):

    fig, axs = plt.subplots(2, 2, figsize=(12, 10))

    target_uniq = data[target].unique()

    axs[0, 0].set_title("Distribution of target for target=" + str(target_uniq[0]))
    sns.histplot(
        data=data[data[target] == target_uniq[0]],
        x=predictor,
        kde=True,
        ax=axs[0, 0],
        color="teal",
        stat="density",
    )

    axs[0, 1].set_title("Distribution of target for target=" + str(target_uniq[1]))
    sns.histplot(
        data=data[data[target] == target_uniq[1]],
        x=predictor,
        kde=True,
        ax=axs[0, 1],
        color="orange",
        stat="density",
    )

    axs[1, 0].set_title("Boxplot w.r.t target")
    sns.boxplot(data=data, x=target, y=predictor, ax=axs[1, 0], palette="gist_rainbow")

    axs[1, 1].set_title("Boxplot (without outliers) w.r.t target")
    sns.boxplot(
        data=data,
        x=target,
        y=predictor,
        ax=axs[1, 1],
        showfliers=False,
        palette="gist_rainbow",
    )

    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(15, 7))
sns.heatmap(loan.corr(numeric_only=True), annot=True, vmin=-1, vmax=1, fmt=".2f", cmap="Spectral") # Complete the code to get the heatmap of the data
plt.show()

- We see the correlation between Income and CCAvg, which is average spending on credit cards per month.
- We also see correlation between Personal loan and income of the individual.
- We see correlation between CCAvg and Personal loan. Customer's ability to take personal loan is also dependent on average spending on credit cards per month.
- There is a correlation between Securities account and Certificate of Deposit(CD) account for the customer , which is 0.32.

In [ ]:
#Correlation between customer's interest in purchasing a loan varies with their education
stacked_barplot(loan, "Education", "Personal_Loan")

- Customers who have advanced or professional degree(Education) are more likely to take a personal loan.
- Customers with graduate degree seem to be the next who take personal load and the ones with Undergrad education seem to be the last in numbers to take personal loan.

In [ ]:
## Correlation to plot stacked barplot for Personal Loan and Family
stacked_barplot(loan,"Personal_Loan","Family")

- Customers with more number of family members seem to be taking a personal loan than when there are 1 or 2 family members.

In [ ]:
## Code to plot stacked barplot for Personal Loan and Securities_Account
stacked_barplot(loan,"Personal_Loan","Securities_Account")

- Customers who have securities account , tend to take personal loans over customers who do not have securities account.

In [ ]:
## Code to plot stacked barplot for Personal Loan and CD_Account
stacked_barplot(loan,"Personal_Loan","CD_Account")

- From the above visualization , we see that there is not significant relation between the customers who have Certificates of deposit account to taking personal loan. Though the number is slightly more for customers who have CD_Account , the customers with no CD_account is still higher in taking a personal loan.

In [ ]:
## Stacked barplot for Personal Loan and Online
stacked_barplot(loan,"Personal_Loan","Online")

- From the above visualization, we notice that there is not a significant difference between customers who have online account vs who don't have online account that drives their decision to take personal loan.

In [ ]:
## Stacked barplot for Personal Loan and CreditCard
stacked_barplot(loan,"Personal_Loan","CreditCard")

- There is not significant difference between customers who can take personal loan based on whether they have/do not have credit cards. Having credit cards does not affect this decision.

In [ ]:
stacked_barplot(loan,"Personal_Loan","Age")

As we see above the data visualization is complex and shows too many values. We would want to categorize age into different groups to see if it works better.

In [ ]:
# Define the bin edges
groups = [20, 30, 40, 50, 60, 70]

# Create bin labels
labels = ['20-29', '30-39', '40-49', '50-59', '60-69']

loan['Age_Group'] = pd.cut(loan['Age'], bins=groups, labels=labels, right=False)
loan['Age_Group']

In [ ]:
stacked_barplot(loan,"Personal_Loan","Age_Group")

- Based on the above visualization , we see the highest number of customers who take personal loan are between the ages 30-39 followed by 50-59 and 40-49 age groups. This is typically the ages where customers have income and the possibility reduces between the age groups 20-29 and 60-69.

In [ ]:
#Stacked barplot for Personal Loan and ZIPCode
stacked_barplot(loan,"Personal_Loan","ZIPCode")

In [ ]:
#To check how a customer's interest in purchasing a loan varies with their age
distribution_plot_wrt_target(loan, "Age", "Personal_Loan")

In [ ]:
## The code to plot stacked barplot for Personal Loan and Experience
distribution_plot_wrt_target(loan, "Personal_Loan", "Experience")

In [ ]:
# the code to plot stacked barplot for Personal Loan and Income
distribution_plot_wrt_target(loan, "Personal_Loan", "Income")

In [ ]:
#The code to plot stacked barplot for Personal Loan and CCAvg
distribution_plot_wrt_target(loan,"Personal_Loan","CCAvg")

The above visualization helps us understand how the distribution of average credit card spending differs between customers who took a personal loan and those who didn't, providing insights into the potential relationship between these two variables.

##Outlier detection and treatment

In [ ]:
Q1 = loan.select_dtypes(include=["float64", "int64"]).quantile(0.25)  # To find the 25th percentile and 75th percentile.
Q3 = loan.select_dtypes(include=["float64", "int64"]).quantile(0.75)

IQR = Q3 - Q1  # Inter Quantile Range (75th perentile - 25th percentile)

lower = (
    Q1 - 1.5 * IQR
)  # Finding lower and upper bounds for all values. All values outside these bounds are outliers
upper = Q3 + 1.5 * IQR

(
    (loan.select_dtypes(include=["float64", "int64"]) < lower)
    | (loan.select_dtypes(include=["float64", "int64"]) > upper)
).sum() / len(loan) * 100

We see some number of outliers for Securities account , Personal loan , CCAvg , Mortgage and Income. There are no outliers for other variables.

In [ ]:
# dropping Experience as it is perfectly correlated with Age
X = loan.drop(["Personal_Loan", "Experience","Age_Group"], axis=1)
Y = loan["Personal_Loan"]

X = pd.get_dummies(X, columns=["ZIPCode", "Education"], drop_first=True)

X = X.astype(float)

# Splitting data in train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.30, random_state=1
)

In [ ]:
print("Shape of Training set : ", X_train.shape)
print("Shape of test set : ", X_test.shape)
print("Percentage of classes in training set:")
print(y_train.value_counts(normalize=True))
print("Percentage of classes in test set:")
print(y_test.value_counts(normalize=True))

## Model Building

### Model Evaluation Criterion

* Some common methods below to evaluare accuracy, precision , recall and F1 score.


In [ ]:
# function to compute different metrics to check performance of a classification model built using sklearn
def model_performance_classification_sklearn(model, predictors, target):
    """
    Function to compute different metrics to check classification model performance

    model: classifier
    predictors: independent variables
    target: dependent variable
    """

    # predicting using the independent variables
    pred = model.predict(predictors)

    acc = accuracy_score(target, pred)  # to compute Accuracy
    recall = recall_score(target, pred)  # to compute Recall
    precision = precision_score(target, pred)  # to compute Precision
    f1 = f1_score(target, pred)  # to compute F1-score

    # creating a dataframe of metrics
    df_perf = pd.DataFrame(
        {"Accuracy": acc, "Recall": recall, "Precision": precision, "F1": f1,},
        index=[0],
    )

    return df_perf

In [ ]:
#confusion_matrix_sklearn function to create/plot confusion matrix.
def confusion_matrix_sklearn(model, predictors, target):
    """
    To plot the confusion_matrix with percentages

    model: classifier
    predictors: independent variables
    target: dependent variable
    """
    y_pred = model.predict(predictors)
    cm = confusion_matrix(target, y_pred)
    labels = np.asarray(
        [
            ["{0:0.0f}".format(item) + "\n{0:.2%}".format(item / cm.flatten().sum())]
            for item in cm.flatten()
        ]
    ).reshape(2, 2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=labels, fmt="")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")

In [ ]:
model = DecisionTreeClassifier(criterion="gini", random_state=1)
model.fit(X_train, y_train)

In [ ]:
confusion_matrix_sklearn(model, X_train, y_train)

In [ ]:
decision_tree_perf_train = model_performance_classification_sklearn(
    model, X_train, y_train
)
decision_tree_perf_train

In [ ]:
#Visualizing the tree with train set
feature_names = list(X_train.columns)
print(feature_names)

In [ ]:
plt.figure(figsize=(20, 30))
out = tree.plot_tree(
    model,
    feature_names=feature_names,
    filled=True,
    fontsize=9,
    node_ids=False,
    class_names=None,
)
# below code will add arrows to the decision tree split if they are missing
for o in out:
    arrow = o.arrow_patch
    if arrow is not None:
        arrow.set_edgecolor("black")
        arrow.set_linewidth(1)
plt.show()

In [ ]:
#Displaying the rules of a decision tree -

print(tree.export_text(model, feature_names=feature_names, show_weights=True))

In [ ]:
# Code to print importance of features in the tree building
print(
    pd.DataFrame(
        model.feature_importances_, columns=["Imp"], index=X_train.columns
    ).sort_values(by="Imp", ascending=False)
)

In [ ]:
importances = model.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(8, 8))
plt.title("Feature Importances")
plt.barh(range(len(indices)), importances[indices], color="violet", align="center")
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel("Relative Importance")
plt.show()

-  The above visualization shows quite a few important features affecting customer's decision to take Personal loan in training dat.
- Income being the highest feature followed by family , higher education, CCAvg and so on.

In [ ]:
#Confusion matrix to check test data
confusion_matrix_sklearn(model, X_test, y_test)

In [ ]:
decision_tree_perf_test = model_performance_classification_sklearn(
    model, X_test, y_test
)
decision_tree_perf_test

Accuracy percentage is higher and is 98% with Recall 89% , precision being 93% and F1 Score 91%.

In [ ]:
#Visualizing the tree with test set
feature_names_test = list(X_test.columns)
print(feature_names_test)

In [ ]:
plt.figure(figsize=(20, 30))
out = tree.plot_tree(
    model,
    feature_names=feature_names_test,
    filled=True,
    fontsize=9,
    node_ids=False,
    class_names=None,
)
# below code will add arrows to the decision tree split if they are missing
for o in out:
    arrow = o.arrow_patch
    if arrow is not None:
        arrow.set_edgecolor("black")
        arrow.set_linewidth(1)
plt.show()

## Model Performance Improvement

**Pre-pruning**


In [ ]:
# Define the parameters of the tree to iterate over
max_depth_values = np.arange(2, 7, 2)
max_leaf_nodes_values = [50, 75, 150, 250]
min_samples_split_values = [10, 30, 50, 70]

# Initialize variables to store the best model and its performance
best_estimator = None
best_score_diff = float('inf')
best_test_score = 0.0

# Iterate over all combinations of the specified parameter values
for max_depth in max_depth_values:
    for max_leaf_nodes in max_leaf_nodes_values:
        for min_samples_split in min_samples_split_values:

            # Initialize the tree with the current set of parameters
            estimator = DecisionTreeClassifier(
                max_depth=max_depth,
                max_leaf_nodes=max_leaf_nodes,
                min_samples_split=min_samples_split,
                class_weight='balanced',
                random_state=42
            )

            # Fit the model to the training data
            estimator.fit(X_train, y_train)

            # Make predictions on the training and test sets
            y_train_pred = estimator.predict(X_train)
            y_test_pred = estimator.predict(X_test)

            # Calculate recall scores for training and test sets
            train_recall_score = recall_score(y_train, y_train_pred)
            test_recall_score = recall_score(y_test, y_test_pred)

            # Calculate the absolute difference between training and test recall scores
            score_diff = abs(train_recall_score - test_recall_score)

            # Update the best estimator and best score if the current one has a smaller score difference
            if (score_diff < best_score_diff) & (test_recall_score > best_test_score):
                best_score_diff = score_diff
                best_test_score = test_recall_score
                best_estimator = estimator

# Print the best parameters
print("Best parameters found:")
print(f"Max depth: {best_estimator.max_depth}")
print(f"Max leaf nodes: {best_estimator.max_leaf_nodes}")
print(f"Min samples split: {best_estimator.min_samples_split}")
print(f"Best test recall score: {best_test_score}")

In [ ]:
estimator = best_estimator
estimator.fit(X_train,y_train)

In [ ]:
#Confusion matrix to train the data
confusion_matrix_sklearn(estimator, X_train, y_train)

In [ ]:
decision_tree_tune_perf_train = model_performance_classification_sklearn(
    estimator, X_train, y_train
)
decision_tree_tune_perf_train

In [ ]:
plt.figure(figsize=(10, 10))
out = tree.plot_tree(
    estimator,
    feature_names=feature_names,
    filled=True,
    fontsize=9,
    node_ids=False,
    class_names=None,
)
# below code will add arrows to the decision tree split if they are missing
for o in out:
    arrow = o.arrow_patch
    if arrow is not None:
        arrow.set_edgecolor("black")
        arrow.set_linewidth(1)
plt.show()

- Pre pruned data seem to create a simpler tree , great for generalization.

In [ ]:
#Rules of a decision tree
print(tree.export_text(estimator, feature_names=feature_names, show_weights=True))

In [ ]:
#Code for feature importance
print(
    pd.DataFrame(
        estimator.feature_importances_, columns=["Imp"], index=X_train.columns
    ).sort_values(by="Imp", ascending=False)
)

In [ ]:
importances = estimator.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(8, 8))
plt.title("Feature Importances")
plt.barh(range(len(indices)), importances[indices], color="violet", align="center")
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel("Relative Importance")
plt.show()

- Highest importance seem to be on Income of the individual followed by CCAvg and number of members in a family.

#Checking performance on Test data

In [ ]:
confusion_matrix_sklearn(model,X_test,y_test)

In [ ]:
decision_tree_perf_test = model_performance_classification_sklearn(
    model, X_test, y_test
)
decision_tree_perf_test



#    **Post Pruning**

In [ ]:
#The below code is generating the necessary information to determine which ccp_alpha values are relevant for pruning the decision tree.
#It also tells us how the impurity changes as the tree is pruned at these alpha values.
clf = DecisionTreeClassifier(random_state=1)
path = clf.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

In [ ]:
pd.DataFrame(path)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ccp_alphas[:-1], impurities[:-1], marker="o", drawstyle="steps-post")
ax.set_xlabel("effective alpha")
ax.set_ylabel("total impurity of leaves")
ax.set_title("Total Impurity vs effective alpha for training set")
plt.show()

The above plot helps in understanding how the impurity of the decision tree changes as you increase the ccp_alpha value, which is a key parameter for post-pruning. Higher ccp_alpha values lead to more pruning and thus creating simpler trees. This can help prevent overfitting.

In [ ]:
clfs = []
for ccp_alpha in ccp_alphas:
    clf = DecisionTreeClassifier(random_state=1, ccp_alpha=ccp_alpha)
    clf.fit(X_train,y_train)     ## Complete the code to fit decision tree on training data
    clfs.append(clf)
print(
    "Number of nodes in the last tree is: {} with ccp_alpha: {}".format(
        clfs[-1].tree_.node_count, ccp_alphas[-1]
    )
)

In [ ]:
clfs = clfs[:-1]
ccp_alphas = ccp_alphas[:-1]

node_counts = [clf.tree_.node_count for clf in clfs]
depth = [clf.tree_.max_depth for clf in clfs]
fig, ax = plt.subplots(2, 1, figsize=(10, 7))
ax[0].plot(ccp_alphas, node_counts, marker="o", drawstyle="steps-post")
ax[0].set_xlabel("alpha")
ax[0].set_ylabel("number of nodes")
ax[0].set_title("Number of nodes vs alpha")
ax[1].plot(ccp_alphas, depth, marker="o", drawstyle="steps-post")
ax[1].set_xlabel("alpha")
ax[1].set_ylabel("depth of tree")
ax[1].set_title("Depth vs alpha")
fig.tight_layout()

In [ ]:
#We can compare Recall vs alpha for training and testing sets
recall_train = []
for clf in clfs:
    pred_train = clf.predict(X_train)
    values_train = recall_score(y_train, pred_train)
    recall_train.append(values_train)

recall_test = []
for clf in clfs:
    pred_test = clf.predict(X_test)
    values_test = recall_score(y_test, pred_test)
    recall_test.append(values_test)

In [ ]:
#Then plot a visualization for comparison
fig, ax = plt.subplots(figsize=(15, 5))
ax.set_xlabel("alpha")
ax.set_ylabel("Recall")
ax.set_title("Recall vs alpha for training and testing sets")
ax.plot(ccp_alphas, recall_train, marker="o", label="train", drawstyle="steps-post")
ax.plot(ccp_alphas, recall_test, marker="o", label="test", drawstyle="steps-post")
ax.legend()
plt.show()

- Training set recall seem to decrease as alpha values increase for training set. Similar pattern is seen for Test set.

In [ ]:
index_best_model = np.argmax(recall_test)
best_model = clfs[index_best_model]
print(best_model)

In [ ]:
 ## Code by adding the correct ccp_alpha value
estimator_2 = DecisionTreeClassifier(
    ccp_alpha=ccp_alpha, class_weight={0: 0.15, 1: 0.85}, random_state=1
)
estimator_2.fit(X_train, y_train)

We want to check performance on Training data as a next step

In [ ]:
confusion_matrix_sklearn(estimator_2,X_train,y_train)

In [ ]:
decision_tree_tune_post_train = model_performance_classification_sklearn(model,X_train, y_train_pred) ## Complete the code to check performance on train data
decision_tree_tune_post_train

In [ ]:
plt.figure(figsize=(10, 10))
out = tree.plot_tree(
    estimator_2,
    feature_names=feature_names,
    filled=True,
    fontsize=9,
    node_ids=False,
    class_names=None,
)
# below code will add arrows to the decision tree split if they are missing
for o in out:
    arrow = o.arrow_patch
    if arrow is not None:
        arrow.set_edgecolor("black")
        arrow.set_linewidth(1)
plt.show()

- This decision tree seem to be too simplified and small. Reason could be aggressive pruning and removing too many features.

In [ ]:
# Text report showing the rules of a decision tree -
print(tree.export_text(estimator_2, feature_names=feature_names, show_weights=True))

In [ ]:
# importance of features in the tree building . Feature importance will help us understand model and feature selection.
#Important features will help us give more focus on data collection and preprocessing.
print(
    pd.DataFrame(
        estimator_2.feature_importances_, columns=["Imp"], index=X_train.columns
    ).sort_values(by="Imp", ascending=False)
)

In [ ]:
importances = estimator_2.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(8, 8))
plt.title("Feature Importances")
plt.barh(range(len(indices)), importances[indices], color="violet", align="center")
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel("Relative Importance")
plt.show()

- We still see Income is the important feature that affects customers personal loan decision . However other features seem to be removed in post pruning.

In [ ]:
confusion_matrix_sklearn(model,X_test,y_test)

In [ ]:
decision_tree_tune_post_test = model_performance_classification_sklearn(estimator_2, X_test, y_test) ## Complete the code to get the model performance on test data
decision_tree_tune_post_test

## Model Performance Comparison and Final Model Selection

In [ ]:
# training performance comparison

models_train_comp_df = pd.concat(
    [decision_tree_perf_train.T, decision_tree_tune_perf_train.T, decision_tree_tune_post_train.T], axis=1,
)
models_train_comp_df.columns = ["Decision Tree (sklearn default)", "Decision Tree (Pre-Pruning)", "Decision Tree (Post-Pruning)"]
print("Training performance comparison:")
models_train_comp_df

For Training set , we see Overfitting and positive role of pruning on training data.
- Unpruned tree shows higher accuracy of 98% on the training data and is likely overfit. Pre-pruning can prevent overfitting but too much may lead to underfitting. However, post-pruning appears to be better resulting in a model that performs well on the training data and is likely to generalize better to new, unseen data.
- Recall- original Decisition tree seem to have better recall of 93%. Pre-pruning recall appears to be a red flag and suggests Overfitting. 72% recall after post-pruning suggests a simpler model, which is likely to generalize better to unseen data.
- Precision- Precision after prepruning of 31% indicated the pruning was too aggressive and resulted in underfitting.Tree might be too simple and not able to catch the pattern.Pecision of 98% after post pruning suggests pruning is helpful and show good balance of performance and complexity on the training data.
- F1 score is 92% for unpruned tree which means model is good in identifying positive cases. Very low score of 47% suggests high pre pruning , didn't capture underlying patterns properly.83% of after post pruning more balanced model compared to pre-pruned model.

In [ ]:
# testing performance comparison

models_test_comp_df = pd.concat(
    [decision_tree_perf_test.T, decision_tree_perf_test.T, decision_tree_tune_post_test.T], axis=1,
)
models_test_comp_df.columns = ["Decision Tree (sklearn default)", "Decision Tree (Pre-Pruning)", "Decision Tree (Post-Pruning)"]
print("Test set performance comparison:")
models_test_comp_df

In Test set, we see
- Accuracy- Unpruned tree shows overfitting . After pre-pruning, we see value of 98% , which means the model is preforming better. However , significant reduction after post pruning , suggests the model is underfitting and might have too aggressively pruned.
- Recall - Unpruned decision tree has 93% recall , which is good ability to identify positive cases. After pre pruning , recall is 93% which is good in making decision tree simpler. Post pruning , recall drops to 90% likely a result of simplifying the model to improve its generalization capabilities and prevent overfitting.
- Precision- Unpruned precision of 92% overfitting leading to lower generalization of the unseen data. Similarly after pre pruning 92% , same level of accuracy is maintained. However significantly lower precision of 34% after post-pruning is concerning and suggests a potential issue with the pruning process or the dataset itself.
- F1 score- The initial decision tree without pruning showed strong performance, with an F1 score of 92%.Pre-pruning successfully maintained this high level of performance, suggesting it prevented overfitting without sacrificing crucial information.
Post-pruning, however, severely impacted the model's ability to generalize to unseen data, reducing the F1 score to 50%.

## Actionable Insights and Business Recommendations


 # Insights
 - We see unpruned model is overfitted and Pruning helps to generalize it better especially post pruning based on Accuracy , Recall and Precision in the training set.
 - Athough pruning is helpful here, there is still overfitted model which might miss to detect underlying patterns in unseen data. To ensure a good balance between model complexity and generalization ability, we can be careful with pruning and do not remove too many important features.
 - Decision tree model of the training data seem to be a better one than the test data. To improve the generalization , bank can use different subsets of data to understand patterns in unseen data.

# Recommendations
 - Income seems to be the important factor that affects the decision to take personal loan.
 - Customer's ability to take personal loan is also dependent on average spending on credit cards per month.Bank should focus on strategies by providing customers with lower interest rates on credit cards.
 - Customer's higher or advanced degree seem to be another factor. Bank can publish blogs, articles to simplify financial topics and provide useful information to encourage customers.
 - To encourage more online presence for availing loans ,bank should offer  seamless online and mobile banking by investing in user-friendly mobile apps, and features like real-time transaction alerts and 24X7 chatbot support to enhance customer satisfaction.
 - Using targeted marketing with liability customers using social media, traditional mail, bank can use data to understand their needs.


___